# Goal: check if themes declared correspond to their declaration

## Extraction

In [6]:
import pandas as pd
import s3fs

fs = s3fs.S3FileSystem(
    endpoint_url="https://minio.lab.sspcloud.fr",
    client_kwargs={"region_name": "us-east-1"},
)

In [7]:
fs.ls("s3://mateomorin/legifrance/")

['mateomorin/legifrance/data',
 'mateomorin/legifrance/documents',
 'mateomorin/legifrance/logs',
 'mateomorin/legifrance/metadata']

In [8]:
accords_folder = "s3://mateomorin/legifrance/data/acco_data_2022/"

def gather_shards(path):
    shards = []
    for file in fs.ls(path):
        shards.append(pd.read_parquet("s3://" + file, filesystem=fs))
    return pd.concat(shards)

accords2022 = gather_shards(accords_folder)

In [9]:
# Colonnes vides ou redondantes

accords2022.drop(columns=["legalStatus", "startDate", "endDate", "nature", "id"], inplace=True)

In [10]:
# Accords non visualisables

col_heures_supp = "Heures supplémentaires (contingent, majoration)"
accords2022[~pd.col("conversion_error").isna()][col_heures_supp].sum()

np.int64(2)

## Filtrage si paterne = heures supplémentaires (ou assimilés)

In [11]:
pattern = r"(?i)\bheures?[\s\w'-]{1,20}?\b(supp(lémentaires?|l?)?|sup)\b"

accords2022["contains_heures_supp"] = accords2022["markdown"].str.contains(pattern)

/tmp/ipykernel_51987/2403468757.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  accords2022["contains_heures_supp"] = accords2022["markdown"].str.contains(pattern)


In [12]:
accords2022.columns

Index(['cid', 'title', 'reference', 'dateSignature', 'dateDiffusion',
       'conforme', 'Accords de méthode (PSE)',
       'Accords de méthode (pénibilité)',
       'Aménagement du temps de travail (modulation, annualisation, cycles)',
       'Autre, précisez', 'Autres dispositions Egalité professionnelle',
       'Autres dispositions de conditions de travail (CHSCT, médecine du travail, politique générale de prévention)',
       'Autres dispositions durée et aménagement du temps de travail',
       'Autres dispositions durée et aménagement du temps de travail ',
       'Autres dispositions emploi', 'Calendrier des négociations',
       'Classifications', 'Commissions paritaires', 'Compte épargne temps',
       'Couverture complémentaire santé - maladie',
       'Dispositifs don de jour et jour de solidarité',
       'Distribution d'actions gratuites',
       'Droit syndical, IRP, expression des salariés',
       'Droit à la déconnexion et outils numériques',
       'Durée collective 

In [13]:
import re
import pandas as pd

# Pattern regex (sans le flag (?i) au début car re.IGNORECASE est passé en argument)
pattern = r"\bheures?[\s\w'-]{1,20}?\b(?:supp(?:lémentaires?|l?)?|sup)\b"


def extraire_paragraphe_hs(texte, filtrer_contingent=True):
    if not isinstance(texte, str):
        return []

    paragraphes = texte.split("\n")
    resultats = []

    for para in paragraphes:
        a_hs = re.search(pattern, para, re.IGNORECASE)
        a_contingent = (
            re.search(r"\bcontingent\b", para, re.IGNORECASE)
            if filtrer_contingent
            else True
        )

        if a_hs and a_contingent:
            resultats.append(para.strip())

    return resultats


# Application sur le DataFrame
accords2022["para_hs"] = accords2022["markdown"].apply(
    lambda x: extraire_paragraphe_hs(x, filtrer_contingent=False)
)

In [14]:
mismatches = accords2022[pd.col("contains_heures_supp") & ~pd.col(col_heures_supp)]

In [15]:
with open("mismatch.md", "w") as f:
    f.write(mismatches["markdown"].sample(1).to_list()[0])

In [16]:
mismatches["para_hs"].sample(10).to_list()

[["Concernant la valeur monétaire d'un jour, les parties ont convenu des dispositions suivantes : 1 jour = salaire de base mensuel brut et éléments fixes de salaire (ex : maintien avantages individuels acquis) + heures supplémentaires structurelles et/ou heures d'équivalence s'il y a eu / 26 (1 mois correspondant à 26 jours ouvrables en moyenne)"],
 ['Les Heures de Récupération pouvant être mises en compte sont celles acquises au titre des heures supplémentaires de l’année N-1 et mises en compte en janvier de'],
 ['**2.3. Heures supplémentaires**',
  "Avant la signature de l'Accord, le temps de travail applicable aux salariés de Substipharm hors cadres dirigeants est de trente-neuf (39) heures hebdomadaires : 35 heures plus 4 heures supplémentaires majorées défiscalisées. Ce régime continue de s'appliquer, selon les modalités définies dans l'article 3, aux Non-Cadres, et aux Cadres qui n'adhéreraient pas au Forfait Jours aux deux dates proposées.",
  "Au-delà du régime applicable chez 

## Filtrage si paterne = contingent (strict)

In [17]:
pattern = r"contingent"

accords2022["contains_contingent"] = accords2022["markdown"].str.contains(pattern)

In [18]:
import re

# Pattern regex (sans le flag (?i) au début car re.IGNORECASE est passé en argument)
pattern = r"contingent"


def extraire_paragraphe_contingent(texte):
    if not isinstance(texte, str):
        return []

    paragraphes = texte.split("\n")
    resultats = []

    for para in paragraphes:
        a_contingent = re.search(pattern, para, re.IGNORECASE)

        if a_contingent:
            resultats.append(para.strip())

    return resultats


# Application sur le DataFrame
accords2022["para_contingent"] = accords2022["markdown"].apply(
    lambda x: extraire_paragraphe_hs(x)
)

In [19]:
mismatches = accords2022[pd.col("contains_contingent") & ~pd.col(col_heures_supp)]

In [20]:
len(accords2022[pd.col("contains_contingent")])

4563

In [21]:
len(accords2022[pd.col("contains_contingent") & pd.col(col_heures_supp)])/len(accords2022[pd.col(col_heures_supp)])

0.7085536547433904

In [22]:
len(accords2022[pd.col("contains_contingent") & ~pd.col(col_heures_supp)])/(len(accords2022[pd.col(col_heures_supp) | pd.col("contains_contingent")]))

0.41545454545454547

## Vérification manuelle

In [26]:
samples = mismatches["docx_path"].sample(10, random_state=42).to_list()
samples

['s3://mateomorin/legifrance/documents/2022/12/ACCOTEXT000047542910.docx',
 's3://mateomorin/legifrance/documents/2022/04/ACCOTEXT000045965260.docx',
 's3://mateomorin/legifrance/documents/2022/02/ACCOTEXT000046754274.docx',
 's3://mateomorin/legifrance/documents/2022/12/ACCOTEXT000046851392.docx',
 's3://mateomorin/legifrance/documents/2022/03/ACCOTEXT000045768131.docx',
 's3://mateomorin/legifrance/documents/2022/10/ACCOTEXT000046384767.docx',
 's3://mateomorin/legifrance/documents/2022/02/ACCOTEXT000045727619.docx',
 's3://mateomorin/legifrance/documents/2022/12/ACCOTEXT000046795071.docx',
 's3://mateomorin/legifrance/documents/2022/05/ACCOTEXT000046116356.docx',
 's3://mateomorin/legifrance/documents/2022/12/ACCOTEXT000047069943.docx']

In [25]:
samples = mismatches["para_contingent"].sample(100, random_state=42).to_list()

with open("mismatch_sample.md", "w") as f:
    f.write("\n\n=========================\n\n".join(["\n".join(sample) for sample in samples]))

## Etude en local

Une étude en local révèle que, bien qu'il y ait une partie significative des accords sur la thématique des heures supplémentaires en dehors du thème Légifrance (ou Daccor), les distributions de variables d'intérêt ont une répartition similaire entre les documents comportant contingents qui ne sont pas dans la thématique, et ceux dans la thématique. 